# 01 -- dim_rookie_prospect Seed

**Purpose**: Build the three pre-draft seed tables:
- `dim_position` and `dim_school` — transformer tables used by all ETL
- `dim_rookie_prospect` — current draft class prospect registry

`dim_rookie_prospect` is a **staging dimension**. Prospects graduate to
`dim_nfl_players` (the central player registry) once they sign their rookie
contracts post-draft and receive a `gsis_id` from the NFL.

**Key**: `player_key` (interim MD5 hash, pre-`gsis_id`)

**Outputs** (local Parquet):
- `data/dim_position.parquet`
- `data/dim_school.parquet`
- `data/dim_rookie_prospect.parquet`

In [ ]:
# !pip install requests
# !pip install nflreadpy

import pandas as pd
import numpy as np
import hashlib
import nflreadpy as nfl
from pathlib import Path
from datetime import datetime, date
from dataclasses import dataclass, field

from thefuzz import fuzz, process

@dataclass
class LeagueConfig:
    draft_year: int = 2026
    total_cap: int = 500_000_000
    num_teams: int = 28
    num_conferences: int = 2
    initial_contract_years: int = 3
    extension_contract_years: int = 3
    fa_minimum_salary: int = 2_000_000
    data_dir: str = "data"
    fuzzy_auto_threshold: int = 90
    fuzzy_review_threshold: int = 70


CFG = LeagueConfig()
DATA = Path(CFG.data_dir)
TODAY = date.today().isoformat()
DATA.mkdir(exist_ok=True)

## 1 — dim_Position (transformer table)

Maps every raw position string you'll encounter across sources to a 
canonical `position_detail`, a `position_group` for filtering, and an
`offense_defense` flag. This replaces the giant if/else chains from 
the old M code.

**Maintenance**: When a new raw position appears from a source, add a row.

In [3]:
# ── Position transformer ────────────────────────────────────────────────────
# Columns:
#   position_raw    : exactly as it appears in the source (case-insensitive match)
#   position_detail : the granular canonical name (e.g., EDGE, CB, IOL)
#   position_group  : rolled-up group for fantasy filtering (QB, RB, WR, TE, DL, LB, DB, OL, ST)
#   side_of_ball    : Offense / Defense / Special Teams
#   side_of_ball_sort_order: 1=Offense, 2=Defense, 3=Special Teams
#   fantasy_relevant: True for positions that score in dynasty leagues

_pos_rows = [
    # ── Offense: Skill ──────────────────────────────────────────────────────
    ("QB",   "QB",   "QB", 1, "Offense", 1, True),
    ("RB",   "RB",   "RB", 2, "Offense", 1, True),
    ("HB",   "RB",   "RB", 2, "Offense", 1, True),   # CBS uses HB
    ("FB",   "FB",   "RB", 3, "Offense", 1, False),
    ("WR",   "WR",   "WR", 4, "Offense", 1, True),
    ("TE",   "TE",   "TE", 5, "Offense", 1, True),

    # ── Offense: Line ───────────────────────────────────────────────────────
    ("OT",   "OT",   "OL", 999, "Offense", 1, False),
    ("OG",   "OG",   "OL", 999, "Offense", 1, False),
    ("OL",   "OL",   "OL", 999, "Offense", 1, False),   # generic
    ("C",    "C",    "OL", 999, "Offense", 1, False),
    ("G",    "OG",   "OL", 999, "Offense", 1, False),   # ESPN uses bare G
    ("T",    "OT",   "OL", 999, "Offense", 1, False),   # ESPN uses bare T
    ("IOL",  "IOL",  "OL", 999, "Offense", 1, False),
    ("OC",   "C",    "OL", 999, "Offense", 1, False),   # some sources
    ("TG",   "OG",   "OL", 999, "Offense", 1, False),   # tackle/guard tweener
    ("GT",   "OG",   "OL", 999, "Offense", 1, False),

    # ── Defense: Line ───────────────────────────────────────────────────────
    ("EDGE", "EDGE", "DL", 6,  "Defense", 2, True),
    ("ED",   "EDGE", "DL", 6,  "Defense", 2, True),   # PFF uses ED
    ("DE",   "DE",   "DL", 6,  "Defense", 2, True),
    ("DT",   "DT",   "DL", 6,  "Defense", 2, True),
    ("DI",   "DI",   "DL", 6,  "Defense", 2, True),   # PFF interior
    ("DL",   "DL",   "DL", 6,  "Defense", 2, True),   # generic
    ("NT",   "NT",   "DL", 6,  "Defense", 2, True),

    # ── Defense: Linebacker ─────────────────────────────────────────────────
    ("LB",   "LB",   "LB", 7,  "Defense", 2, True),
    ("ILB",  "ILB",  "LB", 7,  "Defense", 2, True),
    ("OLB",  "OLB",  "LB", 7,  "Defense", 2, True),
    ("MLB",  "ILB",  "LB", 7,  "Defense", 2, True),
    ("WILL", "OLB",  "LB", 7,  "Defense", 2, True),
    ("SAM",  "OLB",  "LB", 7,  "Defense", 2, True),
    ("MIKE", "ILB",  "LB", 7,  "Defense", 2, True),

    # ── Defense: Secondary ──────────────────────────────────────────────────
    ("CB",   "CB",   "DB", 8,  "Defense", 2, True),
    ("S",    "S",    "DB", 8,  "Defense", 2, True),
    ("SAF",  "S",    "DB", 8,  "Defense", 2, True),
    ("FS",   "FS",   "DB", 8,  "Defense", 2, True),
    ("SS",   "SS",   "DB", 8,  "Defense", 2, True),
    ("DB",   "DB",   "DB", 8,  "Defense", 2, True),   # generic

    # ── Special Teams ───────────────────────────────────────────────────────
    ("K",    "K",    "ST", 999,  "Special Teams", 3, False),
    ("P",    "P",    "ST", 999,  "Special Teams", 3, False),
    ("LS",   "LS",   "ST", 999,  "Special Teams", 3, False),
]

dim_position = pd.DataFrame(
    _pos_rows,
    columns=["position_raw", "position_detail",
             "position_group", "position_sort_order",
             "side_of_ball", "side_of_ball_sort_order", "fantasy_relevant"]
)

dim_position["position_raw"] = dim_position["position_raw"].str.upper().str.strip()

out_path = Path(CFG.data_dir) / "dim_position.parquet"
dim_position.to_parquet(out_path, index=False)
print(f"dim_position: {len(dim_position)} mappings → {out_path}")
dim_position.head(10)

dim_position: 39 mappings → data\dim_position.parquet


,position_raw,position_detail,position_group,position_sort_order,side_of_ball,fantasy_relevant
0,QB,QB,QB,1,Offense,True
1,RB,RB,RB,2,Offense,True
2,HB,RB,RB,2,Offense,True
3,FB,FB,RB,3,Offense,False
4,WR,WR,WR,4,Offense,True
5,TE,TE,TE,5,Offense,True
6,OT,OT,OL,999,Offense,False
7,OG,OG,OL,999,Offense,False
8,OL,OL,OL,999,Offense,False
9,C,C,OL,999,Offense,False


## 2 — dim_School (transformer table)

Maps raw school names across sources to a canonical name and conference.
Seeded from nflverse combine data, then extended as new names appear.

**Maintenance**: When fuzzy-match flags an unknown school, add a row.

In [4]:
# ── School transformer (seed — will grow as sources are added) ──────────────
_school_aliases = {
    # Existing / Corrected Realignment Mappings
    "LSU":                    ("Louisiana State", "SEC"),
    "Louisiana State":        ("Louisiana State", "SEC"),
    "Ole Miss":               ("Mississippi",     "SEC"),
    "Mississippi":            ("Mississippi",     "SEC"),
    "USC":                    ("Southern California", "Big Ten"),
    "Southern California":    ("Southern California", "Big Ten"),
    "Miami":                  ("Miami (FL)",      "ACC"),
    "Miami (FL)":             ("Miami (FL)",      "ACC"),
    "Miami (OH)":             ("Miami (OH)",      "MAC"),
    "Pitt":                   ("Pittsburgh",      "ACC"),
    "Pittsburgh":             ("Pittsburgh",      "ACC"),
    "NC State":               ("North Carolina State", "ACC"),
    "N.C. State":             ("North Carolina State", "ACC"),
    "North Carolina State":   ("North Carolina State", "ACC"),
    "N.C. Tarheels":          ("North Carolina",  "ACC"),
    "North Carolina":         ("North Carolina",  "ACC"),
    "UNC":                    ("North Carolina",  "ACC"),
    "UCF":                    ("Central Florida", "Big 12"),
    "Central Florida":        ("Central Florida", "Big 12"),
    "SMU":                    ("Southern Methodist", "ACC"),
    "Southern Methodist":     ("Southern Methodist", "ACC"),
    "BYU":                    ("Brigham Young",   "Big 12"),
    "Brigham Young":          ("Brigham Young",   "Big 12"),
    "TCU":                    ("Texas Christian", "Big 12"),
    "Texas Christian":        ("Texas Christian", "Big 12"),
    "UTSA":                   ("UT San Antonio",  "AAC"),
    "UT San Antonio":         ("UT San Antonio",  "AAC"),
    "App State":              ("Appalachian State", "Sun Belt"),
    "Appalachian State":      ("Appalachian State", "Sun Belt"),
    "UConn":                  ("Connecticut",     "Independent"),
    "Connecticut":            ("Connecticut",     "Independent"),
    "ECU":                    ("East Carolina",   "AAC"),
    "East Carolina":          ("East Carolina",   "AAC"),
    "UL Monroe":              ("Louisiana-Monroe", "Sun Belt"),
    "Louisiana-Monroe":       ("Louisiana-Monroe", "Sun Belt"),
    "ULL":                    ("Louisiana-Lafayette", "Sun Belt"),
    "Louisiana-Lafayette":    ("Louisiana-Lafayette", "Sun Belt"),
    "Louisiana":              ("Louisiana-Lafayette", "Sun Belt"),
    "UNLV":                   ("Nevada-Las Vegas", "Mountain West"),
    "Nevada-Las Vegas":       ("Nevada-Las Vegas", "Mountain West"),

    # Expanded: Major Realignment & Powerhouses
    "Texas":                  ("Texas",           "SEC"),
    "UT":                     ("Texas",           "SEC"),
    "Oklahoma":               ("Oklahoma",        "SEC"),
    "OU":                     ("Oklahoma",        "SEC"),
    "Oregon":                 ("Oregon",          "Big Ten"),
    "Washington":             ("Washington",      "Big Ten"),
    "Washington (WA)":        ("Washington",      "Big Ten"),
    "Colorado":               ("Colorado",        "Big 12"),
    "Utah":                   ("Utah",            "Big 12"),
    "Stanford":               ("Stanford",        "ACC"),
    "California":             ("California",      "ACC"),
    "Cal":                    ("California",      "ACC"),
    "Arizona":                ("Arizona",         "Big 12"),
    "Arizona State":          ("Arizona State",   "Big 12"),
    "ASU":                    ("Arizona State",   "Big 12"),

    # Expanded: High-Volume Fantasy Factories & Abbreviations
    "Ohio State":             ("Ohio State",      "Big Ten"),
    "Ohio St.":               ("Ohio State",      "Big Ten"),
    "Ohio St":                ("Ohio State",      "Big Ten"),
    "tOSU":                   ("Ohio State",      "Big Ten"),
    "Penn State":             ("Penn State",      "Big Ten"),
    "Penn St.":               ("Penn State",      "Big Ten"),
    "PSU":                    ("Penn State",      "Big Ten"),
    "Michigan":               ("Michigan",        "Big Ten"),
    "Mich.":                  ("Michigan",        "Big Ten"),
    "Alabama":                ("Alabama",         "SEC"),
    "Bama":                   ("Alabama",         "SEC"),
    "Georgia":                ("Georgia",         "SEC"),
    "UGA":                    ("Georgia",         "SEC"),
    "Florida":                ("Florida",         "SEC"),
    "UF":                     ("Florida",         "SEC"),
    "Florida State":          ("Florida State",   "ACC"),
    "Florida St.":            ("Florida State",   "ACC"),
    "FSU":                    ("Florida State",   "ACC"),
    "Texas A&M":              ("Texas A&M",       "SEC"),
    "Texas A & M":            ("Texas A&M",       "SEC"),
    "TAMU":                   ("Texas A&M",       "SEC"),
    "Notre Dame":             ("Notre Dame",      "Independent"),
    "ND":                     ("Notre Dame",      "Independent"),
    "Virginia Tech":          ("Virginia Tech",   "ACC"),
    "VTech":                  ("Virginia Tech",   "ACC"),
    "VT":                     ("Virginia Tech",   "ACC"),
    "Georgia Tech":           ("Georgia Tech",    "ACC"),
    "GT":                     ("Georgia Tech",    "ACC"),
    
    # Expanded: Mid-Majors With Frequent Mismatches
    "Middle Tennessee":       ("Middle Tennessee State", "CUSA"),
    "MTSU":                   ("Middle Tennessee State", "CUSA"),
    "FIU":                    ("Florida International", "CUSA"),
    "Florida International":  ("Florida International", "CUSA"),
    "FAU":                    ("Florida Atlantic", "AAC"),
    "Florida Atlantic":       ("Florida Atlantic", "AAC"),
    "UMass":                  ("Massachusetts",   "Independent"),
    "Massachusetts":          ("Massachusetts",   "Independent"),
}

dim_school_rows = [
    {"school_raw": k, "school_canonical": v[0], "conference": v[1]}
    for k, v in _school_aliases.items()
]

# Create DataFrame
dim_school = pd.DataFrame(dim_school_rows)

# Save to Parquet
out_path = Path(CFG.data_dir) / "dim_school.parquet"
dim_school.to_parquet(out_path, index=False)

print(f"dim_school: {len(dim_school)} alias mappings saved to → {out_path}")

dim_school: 91 alias mappings saved to → data\dim_school.parquet


## 3 — Helper Functions

Reusable parsing/cleaning functions that replace the M code transformations.

In [5]:
def parse_height_to_inches(ht_value) -> float | None:
    """
    Convert any height format to total inches.

    Handles:
      - Already numeric (assume inches if > 12, else feet)
      - "6'2\"" or "6'2" or "6'2"  ->  74
      - '6-2' or '6-02'            ->  74
      - '602' or '510' (NFL compact) ->  72 or 70
      - '6\' 2"' (space after foot) ->  74
      - None / NaN                  ->  None
    """
    if pd.isna(ht_value):
        return None

    # Already a number
    if isinstance(ht_value, (int, float)):
        if ht_value > 12:
            return float(ht_value)  # already inches
        else:
            return float(ht_value) * 12  # feet only

    s = str(ht_value).strip().replace("\\", "").replace('"', '').replace('\u00a0', ' ')

    # Try feet'inches pattern: 6'2, 6' 2
    for sep in ["'", "\u2019"]:
        if sep in s:
            parts = s.split(sep)
            try:
                feet = int(parts[0].strip())
                inches = int(parts[1].strip()) if parts[1].strip() else 0
                return float(feet * 12 + inches)
            except ValueError:
                continue

    # Try feet-inches: 6-2, 6-02
    if "-" in s:
        parts = s.split("-")
        try:
            feet = int(parts[0].strip())
            inches = int(parts[1].strip())
            return float(feet * 12 + inches)
        except ValueError:
            pass

    # Try NFL compact: '602' = 6ft 02in, '510' = 5ft 10in
    if s.isdigit() and len(s) == 3:
        feet = int(s[0])
        inches = int(s[1:])
        if inches < 12:
            return float(feet * 12 + inches)

    # Last resort: try direct numeric
    try:
        val = float(s)
        return val if val > 12 else val * 12
    except ValueError:
        return None


def clean_player_name(name: str) -> str:
    """
    Normalize a player name for matching.

    - Strip periods, special chars, extra whitespace
    - Normalize curly apostrophes and backticks to straight apostrophe
    - Lowercase for comparison

    Returns cleaned name (lowercase). Original casing preserved separately.
    """
    if pd.isna(name):
        return ""
    s = str(name).strip()
    # Remove periods (J.K. Dobbins -> JK Dobbins)
    s = s.replace(".", "")
    # Remove non-breaking spaces
    s = s.replace("\u00a0", " ")
    # Normalize quotes and apostrophes
    s = s.replace("\u2018", "'").replace("\u2019", "'").replace("`", "'")
    # Collapse multiple spaces
    s = " ".join(s.split())
    return s.lower()


def generate_player_key(name: str, position: str, school: str) -> str:
    """Deterministic 12-char player key from name + position + school for dedup."""
    raw = f"{clean_player_name(name)}|{str(position).upper().strip()}|{str(school).strip().lower()}"
    return hashlib.md5(raw.encode()).hexdigest()[:12]


# Quick smoke test
assert parse_height_to_inches("6'2\"") == 74.0
assert parse_height_to_inches("6-2") == 74.0
assert parse_height_to_inches("602") == 74.0
assert parse_height_to_inches(74) == 74.0
assert clean_player_name("D'Andre Swift") == "d'andre swift"
assert clean_player_name("J.K. Dobbins") == "jk dobbins"
print("Helper functions: all smoke tests passed")

Helper functions: all smoke tests passed


## 4 -- Pull nflverse Combine Data (Prospect Seed)

Foundation of `dim_rookie_prospect`. Every combine invitee for `CFG.draft_year`
gets a row, even if they opted out of all drills.

In [6]:
# ── Pull combine data via nflreadpy ─────────────────────────────────────────
# nflreadpy returns Polars DataFrames; convert to pandas immediately
print("Loading combine data via nflreadpy...")
try:
    combine_all = nfl.load_combine(seasons=True).to_pandas()
except Exception as e:
    raise RuntimeError(f"Failed to load nflverse combine data: {e}") from e

print(f"Total combine records (all years): {len(combine_all):,}")
print(f"Columns: {list(combine_all.columns)}")
print(f"Year range: {combine_all['season'].min()} – {combine_all['season'].max()}")
print(f"{CFG.draft_year} invitees: {len(combine_all[combine_all['season'] == CFG.draft_year])}")

combine_current = combine_all[combine_all["season"] == CFG.draft_year].copy()
combine_current.head(5)

Loading combine data via nflreadpy...
Total combine records (all years): 8,968
Columns: ['season', 'draft_year', 'draft_team', 'draft_round', 'draft_ovr', 'pfr_id', 'cfb_id', 'player_name', 'pos', 'school', 'ht', 'wt', 'forty', 'bench', 'vertical', 'broad_jump', 'cone', 'shuttle']
Year range: 2000 – 2026
2026 invitees: 319


,season,draft_year,draft_team,draft_round,draft_ovr,pfr_id,cfb_id,player_name,pos,school,ht,wt,forty,bench,vertical,broad_jump,cone,shuttle
8649,2026,NaN,NaN,NaN,NaN,AdamCh01,chris-adams-2,Chris Adams,OT,Memphis,6-5,311.0,NaN,NaN,NaN,NaN,NaN,NaN
8650,2026,NaN,NaN,NaN,NaN,NaN,joey-aguilar-1,Jose Aguilar,QB,Tennessee,6-3,229.0,NaN,NaN,NaN,NaN,NaN,NaN
8651,2026,NaN,NaN,NaN,NaN,AllaDr00,drew-allar-1,Drew Allar,QB,Penn St.,6-5,228.0,NaN,NaN,NaN,NaN,NaN,NaN
8652,2026,NaN,NaN,NaN,NaN,NaN,cj-allen-1,CJ Allen,LB,Georgia,6-1,230.0,NaN,NaN,NaN,NaN,NaN,NaN
8653,2026,NaN,NaN,NaN,NaN,AlleKa01,kaytron-allen-1,Kaytron Allen,RB,Penn St.,5-11,216.0,NaN,NaN,NaN,NaN,NaN,NaN


## 5 -- Build dim_rookie_prospect from Combine Seed

One row per combine invitee. `player_key` is the interim PK (MD5 hash of
name + position + school) used across fact tables pre-draft.
`gsis_id` is null at seed time; populated post-signing via ETL join against
`dim_nfl_players` on `pfr_id`.

In [7]:
def build_dim_rookie_prospect(combine_df: pd.DataFrame, draft_year: int) -> pd.DataFrame:
    # Filter to current draft year only
    df = combine_df[combine_df["season"] == draft_year].copy()

    if len(df) == 0:
        print(f"Warning: No combine data for {draft_year}. Returning empty frame.")
        return pd.DataFrame()

    # -- Load transformer tables ------------------------------------------
    pos_map    = pd.read_parquet(DATA / "dim_position.parquet")
    school_map = pd.read_parquet(DATA / "dim_school.parquet")

    # -- Normalize raw fields ---------------------------------------------
    df["player_name"]       = df["player_name"].str.strip()
    df["player_name_clean"] = df["player_name"].apply(clean_player_name)
    df["position_raw"]      = df["pos"].str.upper().str.strip()
    df["school_raw"]        = df["school"].str.strip()

    # -- Position lookup --------------------------------------------------
    df = df.merge(
        pos_map[["position_raw", "position_detail", "position_group",
                 "side_of_ball", "fantasy_relevant"]],
        on="position_raw", how="left"
    )
    unmatched = df[df["position_detail"].isna()]["position_raw"].unique()
    if len(unmatched):
        print(f"Warning: unmatched positions (add to dim_position): {list(unmatched)}")

    # -- School lookup ----------------------------------------------------
    df = df.merge(
        school_map.rename(columns={"school_raw": "school_raw_lookup"}),
        left_on="school_raw", right_on="school_raw_lookup", how="left"
    )
    df["school_canonical"] = df["school_canonical"].fillna(df["school_raw"])
    df["conference"]       = df["conference"].fillna("Unknown")
    df.drop(columns=["school_raw_lookup"], inplace=True, errors="ignore")

    # -- Height -----------------------------------------------------------
    df["height_inches"] = df["ht"].apply(parse_height_to_inches)

    # -- Interim player key (MD5 hash of name + position + school) --------
    df["player_key"] = df.apply(
        lambda r: generate_player_key(r["player_name"], r["position_raw"], r["school_raw"]),
        axis=1
    )

    # -- Select final columns ---------------------------------------------
    result = df[[
        "player_key",
        "player_name", "player_name_clean",
        "position_raw", "position_detail", "position_group",
        "side_of_ball", "fantasy_relevant",
        "school_raw", "school_canonical", "conference",
        "height_inches", "wt",
        "pfr_id", "cfb_id",
    ]].copy()
    result.rename(columns={"wt": "weight"}, inplace=True)

    # gsis_id: null at seed time; ETL populates after rookie signing
    # via pfr_id join against dim_nfl_players
    result["gsis_id"]    = pd.NA
    result["draft_year"] = draft_year
    result["source"]     = "nflverse_combine"
    result["added_date"] = TODAY

    result.drop_duplicates(subset=["player_key"], inplace=True)
    return result.reset_index(drop=True)


dim_rookie_prospect = build_dim_rookie_prospect(combine_all, CFG.draft_year)
print(f"dim_rookie_prospect: {len(dim_rookie_prospect)} prospects for {CFG.draft_year}")
print(f"Position groups:\n{dim_rookie_prospect['position_group'].value_counts().to_string()}")
print(f"Fantasy-relevant: {dim_rookie_prospect['fantasy_relevant'].sum()}")

out_path = DATA / "dim_rookie_prospect.parquet"
dim_rookie_prospect.to_parquet(out_path, index=False)
print(f"Saved -> {out_path}")
dim_rookie_prospect.head(10)

dim_rookie_prospect: 319 prospects for 2026
Position groups:
position_group
DL    64
OL    56
DB    54
WR    46
LB    28
TE    27
RB    21
QB    16
ST     7
Fantasy-relevant: 255
Saved -> data\dim_rookie_prospect.parquet


,player_key,player_name,player_name_clean,position_raw,position_detail,position_group,side_of_ball,fantasy_relevant,school_raw,school_canonical,conference,height_inches,weight,pfr_id,cfb_id,gsis_id,draft_year,source,added_date
0,4f61e36da0f6,Chris Adams,chris adams,OT,OT,OL,Offense,False,Memphis,Memphis,Unknown,77.0,311.0,AdamCh01,chris-adams-2,<NA>,2026,nflverse_combine,2026-05-16
1,a912fb2018e4,Jose Aguilar,jose aguilar,QB,QB,QB,Offense,True,Tennessee,Tennessee,Unknown,75.0,229.0,NaN,joey-aguilar-1,<NA>,2026,nflverse_combine,2026-05-16
2,e5417bc86706,Drew Allar,drew allar,QB,QB,QB,Offense,True,Penn St.,Penn State,Big Ten,77.0,228.0,AllaDr00,drew-allar-1,<NA>,2026,nflverse_combine,2026-05-16
3,55c56b655c49,CJ Allen,cj allen,LB,LB,LB,Defense,True,Georgia,Georgia,SEC,73.0,230.0,NaN,cj-allen-1,<NA>,2026,nflverse_combine,2026-05-16
4,85fc4414cd8f,Kaytron Allen,kaytron allen,RB,RB,RB,Offense,True,Penn St.,Penn State,Big Ten,71.0,216.0,AlleKa01,kaytron-allen-1,<NA>,2026,nflverse_combine,2026-05-16
5,3df670e6c8ad,Marcus Allen,marcus allen,CB,CB,DB,Defense,True,North Carolina,North Carolina,ACC,74.0,187.0,AlleMa05,marcus-allen-6,<NA>,2026,nflverse_combine,2026-05-16
6,0fb64b35d726,Luke Altmyer,luke altmyer,QB,QB,QB,Offense,True,Illinois,Illinois,Unknown,74.0,210.0,AltmLu00,luke-altmyer-1,<NA>,2026,nflverse_combine,2026-05-16
7,173d5dc9a360,Aaron Anderson,aaron anderson,WR,WR,WR,Offense,True,LSU,Louisiana State,SEC,68.0,191.0,AndeAa00,aaron-anderson-3,<NA>,2026,nflverse_combine,2026-05-16
8,e025966653e5,David Bailey,david bailey,EDGE,EDGE,DL,Defense,True,Texas Tech,Texas Tech,Unknown,76.0,251.0,BailDa02,david-bailey-5,<NA>,2026,nflverse_combine,2026-05-16
9,f35fadfc88cc,Rueben Bain,rueben bain,EDGE,EDGE,DL,Defense,True,Miami,Miami (FL),ACC,74.0,263.0,BainRu00,rueben-bain-jr-1,<NA>,2026,nflverse_combine,2026-05-16


## 6 -- Validation & Summary

In [8]:
dim_rookie_prospect = pd.read_parquet(DATA / "dim_rookie_prospect.parquet")

print(f"dim_rookie_prospect: {len(dim_rookie_prospect)} prospects for {CFG.draft_year}")
print(f"\nBy position group:")
print(dim_rookie_prospect["position_group"].value_counts().to_string())
print(f"\nBy source:")
print(dim_rookie_prospect["source"].value_counts().to_string())
print(f"\nTop 10 schools:")
print(dim_rookie_prospect["school_canonical"].value_counts().head(10).to_string())
print(f"Missing height : {dim_rookie_prospect['height_inches'].isna().sum()}")
print(f"Missing weight : {dim_rookie_prospect['weight'].isna().sum()}")
print(f"gsis_id status : all null at seed time = {dim_rookie_prospect['gsis_id'].isna().all()}")

# Unmatched positions
unmatched = dim_rookie_prospect[dim_rookie_prospect["position_detail"].isna()]
if len(unmatched):
    print(f"\nWARN: {len(unmatched)} players with unmatched positions:")
    print(unmatched[["player_name", "position_raw"]].to_string())
else:
    print("\nOK: all positions resolved")

dim_rookie_prospect: 319 prospects for 2026

By position group:
position_group
DL    64
OL    56
DB    54
WR    46
LB    28
TE    27
RB    21
QB    16
ST     7

By source:
source
nflverse_combine    319

Top 10 schools:
school_canonical
Texas A&M          13
Alabama            12
Louisiana State    11
Ohio State         11
Georgia            10
Miami (FL)         10
Oklahoma           10
Penn State          9
Florida             9
Clemson             9
Missing height : 0
Missing weight : 0
gsis_id status : all null at seed time = True

OK: all positions resolved
